In [4]:
import os
import json
import numpy as np
from scipy.optimize import linear_sum_assignment

m2 = 0
a2 = 0

def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interW = max(0, xB - xA)
    interH = max(0, yB - yA)
    interArea = interW * interH
    if interArea == 0:
        return 0.0

    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    unionArea = boxAArea + boxBArea - interArea
    return interArea / unionArea


def best_iou_matching(boxesA, boxesB, threshold=0.4):
    if not boxesA or not boxesB:
        return 0

    nA, nB = len(boxesA), len(boxesB)
    iou_matrix = np.zeros((nA, nB))

    for i in range(nA):
        for j in range(nB):
            iou_matrix[i, j] = iou(boxesA[i], boxesB[j])

    row_ind, col_ind = linear_sum_assignment(-iou_matrix)
    matches = sum(iou_matrix[i, j] >= threshold for i, j in zip(row_ind, col_ind))
    return matches


def compute_inter_annotator_agreement(annotation_dir, iou_threshold=0.4):

    annotators = []
    annotator_names = []
    for file in os.listdir(annotation_dir):
        if file.endswith(".json"):
            with open(os.path.join(annotation_dir, file), "r") as f:
                data = json.load(f)
                annotators.append(data["annotations"])
                annotator_names.append(file)
    if len(annotators) < 2:
        raise ValueError("Need at least two annotators to compute agreement.")

    video_folders = sorted(set(a["videoFolder"] for ann in annotators for a in ann))
    print(video_folders)
    per_sample = []

    for folder in video_folders:
        sample_boxes = []
        for ann in annotators:
            boxes = []
            for entry in ann:
                if entry["videoFolder"] == folder:
                    boxes = [g["bbox"] for g in entry["groups"] if g["confidence"] >= 1]
                    break
            sample_boxes.append(boxes)

        #print(sample_boxes)

        global m2
        global a2
        pair_agreements = []
        for i in range(len(sample_boxes)):
            for j in range(i + 1, len(sample_boxes)):
                boxesA, boxesB = sample_boxes[i], sample_boxes[j]
                #print(boxesA)
                #print(boxesB)
                matches = best_iou_matching(boxesA, boxesB, iou_threshold)
                m2+=matches
                avg_boxes = (len(boxesA) + len(boxesB)) / 2 if (len(boxesA) + len(boxesB)) > 0 else 1
                pair_agreements.append(matches / avg_boxes)
                a2+=avg_boxes
        
        #print(pair_agreements)
        agreement = np.mean(pair_agreements) if pair_agreements else 1.0
        if agreement == 0 and sample_boxes[0] == [] and sample_boxes[1] == []:
            agreement = 1
        per_sample.append((folder, agreement))

    avg_agreement = np.mean([a for _, a in per_sample]) if per_sample else 1.0

    print("\nPer-sample Inter-Annotator Agreement (Hungarian Matching):")
    for folder, score in per_sample:
        print(f"  {folder:25s}  {score:.3f}")

    print(f"\nOverall Average Agreement (IoU>{iou_threshold}): {avg_agreement:.3f}")
    print(f"Compared {len(annotators)} annotators: {annotator_names}")

    print(m2/a2)
    
    return per_sample, avg_agreement


if __name__ == "__main__":
    annotation_dir = "annotations"
    compute_inter_annotator_agreement(annotation_dir)


['videos/clip_0001/', 'videos/clip_0002/', 'videos/clip_0003/', 'videos/clip_0004/', 'videos/clip_0005/', 'videos/clip_0006/', 'videos/clip_0007/', 'videos/clip_0008/', 'videos/clip_0009/', 'videos/clip_0010/', 'videos/clip_0011/', 'videos/clip_0012/', 'videos/clip_0013/', 'videos/clip_0014/', 'videos/clip_0015/', 'videos/clip_0016/', 'videos/clip_0017/', 'videos/clip_0018/', 'videos/clip_0019/', 'videos/clip_0020/', 'videos/clip_0021/', 'videos/clip_0022/', 'videos/clip_0023/', 'videos/clip_0024/', 'videos/clip_0025/', 'videos/clip_0026/', 'videos/clip_0027/', 'videos/clip_0028/', 'videos/clip_0029/', 'videos/clip_0030/', 'videos/clip_0031/', 'videos/clip_0032/', 'videos/clip_0033/', 'videos/clip_0034/', 'videos/clip_0035/', 'videos/clip_0036/', 'videos/clip_0037/', 'videos/clip_0038/', 'videos/clip_0039/', 'videos/clip_0040/', 'videos/clip_0041/', 'videos/clip_0042/', 'videos/clip_0043/', 'videos/clip_0044/', 'videos/clip_0045/', 'videos/clip_0046/', 'videos/clip_0047/', 'videos/clip

In [2]:
!pip install scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 10.2 MB/s  0:00:03a 0:00:010:00:0101
